# RQ2: staged ApET and pure random — TextVQA100

The six-question staged calibration passed. Run **all100 frozen TextVQA screening
questions**, including baseline-wrong cases, with full576, native96→32/native288→96,
and three random seeds using each identical schedule. Decoder compression is
**after block16**. There are **900 primary predictions +32 separate controls**.

Use **Internet + T4 ×2** and the same private **rq1-ih-inputs-v1** Dataset.
Push code first. No new inputs or calibration ZIP upload. Native model sources,
validated compression adapter, random streams and the original locked runtime
are unchanged. The compact reviewed calibration reference is included with Git.

Set `RUN_STAGED_TEXTVQA100 = True`, then run cells in order. Default False prevents
background inference on import. Download the final ZIP and matching log.
See `research/RQ2_STAGED_TEXTVQA100.md` for scoring, controls and limitations.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_STAGED_TEXTVQA100 = False  # Set True: all100 questions; nine conditions, 900 primary predictions.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None  # Existing rq1-ih-inputs-v1 Dataset/ZIP; not the older rq1-inputs-v1.
CHECKPOINT_ARGS = []  # Optional existing original-format model and vision paths.

if not RUN_STAGED_TEXTVQA100:
    raise RuntimeError("Set RUN_STAGED_TEXTVQA100 = True, then run cells manually.")
print("Preparing staged TextVQA100 code; model inference has not started.", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True, timeout=120)
if not (repo / "scripts/run_rq2_staged_textvqa100.py").is_file():
    raise RuntimeError("Push the new staged TextVQA100 files first, then update this checkout.")
original_python = repo / "environments/original/.venv/bin/python"
rq2_staged_textvqa_runs = []
print("Repository:", repo, flush=True)


## Prepare the existing input Dataset and locked runtime

This reuses the original environment/cache in a live session. Independent notebook sessions do not share virtual environments or `/tmp` checkpoints.


In [ ]:
sys.path.insert(0, str(repo / "scripts"))
from rq1_input_ih import locate_inputs
inputs = locate_inputs(
    [Path("/kaggle/input"), Path("/kaggle/working")],
    Path("/kaggle/working/rq1-ih-inputs-v1"), explicit=INPUT_ZIP,
)
print("Accepted input directory:", inputs, flush=True)
setup_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
setup_log = Path(f"/kaggle/working/rq2-staged-textvqa-setup-{setup_stamp}.log")
runtime_report = Path(f"/kaggle/working/rq2-staged-textvqa-runtime-{setup_stamp}.json")

def stream_command(command, stream, *, cwd=None):
    print("Running:", " ".join(map(str, command)), flush=True)
    stream.write("COMMAND " + repr(command) + "\n")
    stream.flush()
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", bufsize=1)
    lines = []
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
            lines.append(line)
        status = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    stream.write(f"EXIT {status}\n")
    stream.flush()
    if status:
        raise RuntimeError(f"Command failed (exit {status}); preserve {stream.name} before changing anything.")
    return "".join(lines)

with setup_log.open("x", encoding="utf-8") as stream:
    stream_command([
        "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
        "--output", str(runtime_report),
    ], stream, cwd=repo)
print("Locked original runtime ready. Run the prerequisite/helper cell next.", flush=True)
print("Setup log:", setup_log)
print("Runtime report:", runtime_report)


## Run current prerequisites

The same actual tiny18-block native tests exercise native/direct generation,
random guards, cache geometry and frozen replays. New CPU tests check all100
identities, paired score/seed denominators, compact exports and notebook guards.
All must pass with **zero skips**. No pretrained weights are needed for these tests.


In [ ]:
def verify_rq2_staged_textvqa_prerequisites(*, refresh_runtime=False):
    globals()["rq2_staged_textvqa_prerequisites_passed"] = False
    with setup_log.open("a", encoding="utf-8") as stream:
        if refresh_runtime:
            stream_command([str(original_python), "-X", "utf8", "-I", "scripts/check_original_env.py",
                "--require-cuda", "--expected-gpus", "2", "--output", str(runtime_report)], stream, cwd=repo)
        output = stream_command([str(original_python), "-X", "utf8", "-m", "unittest",
            "research.tests.test_rq2_native", "research.tests.test_rq2_staged_protocol",
            "research.tests.test_rq2_staged_runner", "research.tests.test_rq2_staged_native",
            "research.tests.test_rq2_staged_textvqa_protocol", "research.tests.test_rq2_staged_textvqa_runner", "-v"], stream, cwd=repo)
        counts = re.findall(r"Ran (\d+) tests?", output)
        if not counts or int(counts[-1]) <= 26 or re.search(r"skipped\s*=", output):
            raise RuntimeError("Require all actual staged native tests and zero skips; preserve setup log.")
    command = [str(original_python), "-X", "utf8", "-I", "-c",
        "import json,sys; from pathlib import Path; sys.path.insert(0,str(Path.cwd()/'scripts')); "
        "from run_rq2_staged_textvqa100 import prerequisite_signature,TESTS; "
        "print(json.dumps({'source_sha256':prerequisite_signature(),'test_modules':list(TESTS)}))"]
    signature = json.loads(subprocess.run(command, cwd=repo, capture_output=True, text=True,
                                          check=True, timeout=120).stdout)
    import hashlib
    proof = {"version": "rq2-staged-textvqa100-v1", "status": "passed", "tests_run": int(counts[-1]),
        "skipped": 0, **signature,
        "runtime_report_sha256": hashlib.sha256(runtime_report.read_bytes()).hexdigest(),
        "setup_log_sha256": hashlib.sha256(setup_log.read_bytes()).hexdigest()}
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    globals()["staged_textvqa_prerequisite_report"] = Path(f"/kaggle/working/rq2-staged-textvqa-prerequisites-{stamp}.json")
    staged_textvqa_prerequisite_report.write_text(json.dumps(proof, indent=2) + "\n")
    globals()["rq2_staged_textvqa_prerequisites_passed"] = True
    print("Staged prerequisites passed:", proof["tests_run"], "tests, zero skips.", flush=True)


def run_rq2_staged_textvqa100():
    if not globals().get("rq2_staged_textvqa_prerequisites_passed", False):
        raise RuntimeError("Run verify_rq2_staged_textvqa_prerequisites() successfully before loading pretrained weights.")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    folder = Path(f"/kaggle/working/rq2-staged-textvqa100-v1-{stamp}")
    archive, log = folder.with_suffix(".zip"), folder.with_suffix(".log")
    globals().setdefault("rq2_staged_textvqa_runs", []).append(folder)
    command = [str(original_python), "-X", "utf8", "-I", "scripts/run_rq2_staged_textvqa100.py",
        "--inputs", str(inputs), "--output", str(folder), "--run-textvqa100",
        "--require-cuda", "--expected-gpus", "2", "--download-budget-gib", "20",
        "--setup-log", str(setup_log), "--runtime-report", str(runtime_report),
        "--prerequisite-report", str(staged_textvqa_prerequisite_report), *CHECKPOINT_ARGS]
    print("Run directory:", folder, flush=True)
    print("One model load; all100 questions ×9 conditions =900 primary predictions +32 controls.", flush=True)
    fallback_archive = False
    try:
        with log.open("x", encoding="utf-8") as stream:
            stream_command(command, stream, cwd=repo)
    finally:
        if not archive.exists():
            fallback_archive = True
            fallback = folder / "notebook-fallback"
            fallback.mkdir(parents=True, exist_ok=True)
            (fallback / "README.txt").write_text("Incomplete fallback: normal runner ZIP missing; no successful full100 execution established.\n")
            for path in [setup_log, runtime_report, staged_textvqa_prerequisite_report, log]:
                if path.is_file():
                    shutil.copy2(path, fallback / path.name)
            with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as bundle:
                for path in sorted(folder.rglob("*")):
                    if path.is_file():
                        bundle.write(path, path.relative_to(folder))
        print("Download:", archive, "and", log, flush=True)
        if (folder / "report.json").is_file():
            report = json.loads((folder / "report.json").read_text())
            for key in ("status", "execution_passed", "progress", "wall_seconds", "error"):
                if key in report:
                    print(key + ":", report[key], flush=True)
    if fallback_archive:
        raise RuntimeError("Runner ZIP missing; preserve fallback ZIP/log for diagnosis.")
    report = json.loads((folder / "report.json").read_text())
    if report.get("version") != "rq2-staged-textvqa100-v1" or report.get("status") != "passed" or not report.get("execution_passed"):
        raise RuntimeError("Staged TextVQA100 execution did not pass; download ZIP/log for review.")
    for label, summary in report["summaries"].items():
        print(label + ":", summary["questions"], "questions", flush=True)
        for name, value in summary["conditions"].items():
            print(name, "consensus=", round(value["mean_consensus_score"], 4),
                  "full-credit=", value["full_credit_count"], "APR=", value["APR"], flush=True)
    print("Full100 execution passed. Download ZIP + log for analysis.", flush=True)
    return folder


# Re-executing this cell invalidates the previous test pass.
globals()["rq2_staged_textvqa_prerequisites_passed"] = False


In [ ]:
verify_rq2_staged_textvqa_prerequisites()


## Run all100 questions

The model loads once; all nine conditions run sequentially per question.
A fresh baseline and all baseline-wrong cases are included. Controls add32
generations; they do not increase primary denominators. The ZIP keeps raw
answers, partial consensus credit, per-seed comparisons, preservation, gains/losses,
paired intervals, the 94-question complement and compact diagnostic evidence.


In [ ]:
staged_textvqa100_dir = run_rq2_staged_textvqa100()


## Download ZIP and log before stopping the GPU

Keep **rq2-staged-textvqa100-v1-&lt;timestamp&gt;.zip + matching .log**.
The ZIP includes setup/runtime/test proof and the reviewed calibration reference.
Preserve failed or partial exports too. After both files are downloaded you can
stop the session. We can analyze the results locally before choosing GQA/MMStar runs.


In [ ]:
from IPython.display import FileLink, display
paths = [setup_log, runtime_report, staged_textvqa_prerequisite_report]
for folder in rq2_staged_textvqa_runs:
    paths.extend([folder.with_suffix(".zip"), folder.with_suffix(".log")])
for path in paths:
    if path.exists():
        display(FileLink(str(path)))
